# Great Britain Study 04 — What is a race meeting or fixture?

## Bounded study question

> What does British racing mean by a **meeting** and a **fixture**, and are they the same thing?

This is a reader-facing analytical study of British racing structure.

The study begins with official racing terminology rather than imposing a database-derived definition. In particular, it does **not** assume that races sharing a date and racecourse identity necessarily constitute one official meeting or fixture.

The analytical sequence is evidence-led:

1. establish how the British Horseracing Authority uses the terms **fixture** and **meeting**;
2. determine whether they describe the same conceptual unit;
3. only then investigate how that structure can be identified in Inside Rails data.

### Study controls

The mandatory study references are:

- `docs/STUDY_RESEARCH_PLAYBOOK.md`
- `docs/STUDY_DATABASE_REFERENCE.md`
- `docs/STUDY_DATA_ACCESS.md`
- `docs/STUDY_REVISIT_REGISTER.md`
- `docs/RESEARCH_DATA_SOURCE_REGISTER.md`

Current accepted reader-study database:

`data/processed/database/releases/inside_rails_v4.sqlite3`

Database v4 is read-only for this study.

No database query is required until the official sporting terminology has first been established.

In [2]:
# Study 04 setup
#
# This cell defines the complete execution environment required by the
# analytical cells below. The notebook must not depend on variables or
# imports left behind by an earlier interactive Jupyter session.
#
# Database v4 is the accepted immutable reader-study database.
# It is opened only through the project's governed read-only helper.

from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Use the documented local repository root rather than Path.cwd().
#
# Jupyter may execute this notebook with the notebook directory as the
# current working directory, so repository-relative paths must be anchored
# explicitly to the governed project root.
PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v4.sqlite3"
)


# Fail closed if the documented accepted release is not present.
assert DATABASE.is_file(), f"Accepted Database v4 not found: {DATABASE}"


# Cheap sanity check before running the study.
#
# This verifies that the configured database is Database v4 and that the
# Study 04 GB racecourse-aware view contains the documented 111,634 races.
# It does not perform schema discovery or redefine any governed fields.
with connect_read_only(DATABASE) as connection:
    database_version = connection.execute("PRAGMA user_version").fetchone()[0]
    gb_race_rows = connection.execute(
        """
        SELECT COUNT(*)
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        """
    ).fetchone()[0]

assert database_version == 4, (
    f"Expected accepted Database v4, found user_version={database_version}"
)
assert gb_race_rows == 111_634, (
    f"Expected 111,634 governed GB race rows, found {gb_race_rows:,}"
)

print(f"Database v{database_version} confirmed.")
print(f"Governed GB race occurrences: {gb_race_rows:,}")

Database v4 confirmed.
Governed GB race occurrences: 111,634


## Question 1 — Are a meeting and a fixture the same thing?

### Why this matters

Earlier Inside Rails work sometimes grouped races analytically by date and source course label. That was a useful analytical grouping, not an established definition of a British racing meeting or fixture.

Study 03 has now provided governed racecourse identity, so the remaining question is conceptual rather than merely geographical:

> What is the unit that British racing itself calls a fixture or meeting?

Before constructing such a unit from race occurrences, we must establish the terminology from authoritative BHA evidence.

### Method

Use current BHA Rules, General Instructions, fixture-planning material and other BHA primary sources.

Look for evidence that reveals:

- what constitutes a **fixture**;
- how **meeting** is used;
- whether one meeting can contain more than one fixture;
- whether the BHA also uses *meeting* for a single day's racing;
- which term represents the more precise administrative unit.

Do not infer equivalence merely because the words are sometimes used interchangeably in ordinary racing language.

## Question 1 — Evidence notes

### Evidence 1 — BHA General Instructions: Race Programming Policy

**Source:** British Horseracing Authority, *BHA General Instructions — BHAGI 2.1: Race Programming Policy*  
**Current version:** dated 28 March 2026  
**Relevant location:** paragraph 11, page 2 of the PDF

Paragraph 11 states that a **fixture** must contain at least six programmed races and frames the associated prize-fund condition in terms of **"that day"**.

This establishes that, in current formal BHA race-programming instructions, a fixture is a unit containing a programme of races associated with a particular day's racing.

The same instruction also uses **meeting** elsewhere, including named examples such as the Newmarket Craven Meeting and York Dante Meeting. It therefore does not simply replace the word *meeting* with *fixture* throughout its terminology.

Locator:

https://media.britishhorseracing.com/bha/Rules/BHAGI/Section2_Race_Planning.pdf


### Evidence 2 — A multi-day meeting contains separately identified fixtures

**Source:** British Horseracing Authority, *BHA approves amendment to 2025 Fixture List*  
**Published:** 19 December 2024

The BHA describes Newmarket's Craven event as a **three-day meeting**.

It then treats the constituent dates separately in the Fixture List:

- Tuesday 15 April;
- Wednesday 16 April;
- Thursday 17 April.

The release specifically refers to the Tuesday as a **fixture**, while describing all three days collectively as the **three-day Craven meeting**.

This is direct evidence that **meeting and fixture cannot always denote the same unit**: one named meeting can span multiple dated fixtures.

Locator:

https://www.britishhorseracing.com/press_releases/bha-approves-amendment-to-2025-fixture-list-2/


### Evidence 3 — "Meeting" can also refer to a single dated fixture

**Source:** British Horseracing Authority, *BHA confirms the transfer of two Chelmsford City fixtures to Yarmouth and Southwell*  
**Published:** 29 May 2026

The release begins by describing two Chelmsford events as **fixtures**.

It then describes:

- Thursday 18 June as a **fixture**;
- Sunday 5 July as the following **meeting**.

Both are individual dated race programmes being transferred from Chelmsford.

This shows that BHA prose also uses **meeting** for what the surrounding administrative context treats as an individual fixture.


### Evidence 4 — The same event can be called both a fixture and a meeting

**Source:** British Horseracing Authority, *BHA confirms additional Flat fixture at Wolverhampton on Friday 17 July*  
**Published:** 3 July 2026

The BHA announces an **additional Flat fixture** at Wolverhampton on Friday 17 July.

Later in the same release, when discussing Levy Board funding and Wolverhampton hosting the event, it calls it an **additional meeting**.

This is particularly useful evidence because the referent has not changed: the two terms are being used for the same single day's racing within one BHA release.


### Evidence status

The sources establish several distinct features of BHA usage:

1. **fixture** has a precise administrative/race-programming use for a dated programme of races;
2. **meeting** can describe a multi-day named racing event containing several such fixtures;
3. **meeting** can also be used informally or contextually for a single fixture;
4. the terms therefore overlap in BHA usage but are **not universally interchangeable conceptual units**.

No database-derived definition has been used in reaching these observations.

## Question 1 — What we found

The BHA does **not** use **meeting** and **fixture** as perfectly equivalent technical terms.

### Fixture

**Fixture** is the more precise administrative and race-programming concept.

BHA material treats fixtures as units within the official Fixture List, assigns race programmes to them, moves or transfers them between racecourses, changes their times, and may abandon or replace them.

A fixture therefore appears to be a distinct scheduled unit of racing containing a programme of races.

However, this question has **not yet established the exact attributes that identify one fixture**. In particular, we have not yet proved that `date + racecourse` is sufficient.

### Meeting

**Meeting** is a looser racing term.

BHA usage shows that it can refer to:

- a single fixture or day's racing; or
- a wider named event spanning several fixtures/days.

For example, a multi-day named meeting can contain separately identified fixtures, while BHA prose can also call an individual fixture a meeting.

### Conclusion

For Inside Rails purposes:

> **Fixture should be treated as the candidate administrative entity requiring formal identification. Meeting should not initially be modelled as an equivalent database entity.**

The term **meeting** remains useful reader-facing racing language, but its meaning depends on context.

This avoids forcing a single technical identity onto a term that official racing itself uses at more than one level.

### What this does not establish

Question 1 does **not** yet tell us:

- what uniquely identifies a fixture;
- whether a racecourse can stage more than one fixture on the same date;
- whether time-of-day or session is part of fixture identity;
- what happens to fixture identity when a fixture is transferred to another racecourse;
- what happens when a fixture is postponed or rescheduled;
- whether the races themselves can move independently of the fixture;
- whether named multi-day meetings require any database representation at all.

Those are separate structural questions.

## Question 2 — What makes one fixture distinct from another?

### Why this follows from Question 1

Question 1 established that **fixture** is the more useful administrative unit for structural modelling.

The next task is therefore not to count fixtures, but to determine their identity.

A tempting analytical assumption would be:

`racecourse + date = fixture`

That must not be adopted without evidence.

We need to establish whether British racing permits:

- multiple fixtures at the same racecourse on the same date;
- fixture identity to survive a change of racecourse;
- fixture identity to survive a change of date;
- distinctions based on afternoon/evening or another session concept.

### Method

Use BHA fixture-planning and fixture-change evidence to find boundary cases.

Boundary cases are more informative than ordinary fixtures because they reveal which properties can change while the BHA still treats something as the same fixture, and which properties distinguish two fixtures from each other.

## Question 2 — Evidence notes

### Evidence 1 — More than one fixture can exist at the same racecourse on the same date

**Source:** British Horseracing Authority, *Southwell fixtures rearranged for January*  
**Published:** 3 December 2012

Following flooding at Southwell, the BHA transferred its 4 January all-weather fixture to Wolverhampton.

The BHA stated that this fixture would be staged at Wolverhampton in the **afternoon**, with a gap before Wolverhampton's **existing evening fixture**.

Its reallocation table explicitly described the arrangement as:

> Double header with Wolverhampton's evening fixture

This is direct evidence that two separate fixtures can occupy:

- the same racecourse;
- on the same calendar date;
- but in different sessions.

Therefore:

`date + racecourse`

is **not a universally valid definition of fixture identity**.

Important limitation:

This example dates from 2012, before the Inside Rails source period beginning in 2015. It establishes a structural precedent in BHA fixture administration but does not by itself prove that such a double-header occurs within the Inside Rails analytical period.

Locator:

https://www.britishhorseracing.com/press_releases/southwell-fixtures-rearranged-for-january/


### Evidence 2 — A fixture can change racecourse

**Source:** British Horseracing Authority, *BHA confirms the transfer of three Chelmsford City fixtures*  
**Published:** 8 July 2026

The BHA described three Chelmsford City fixtures as being **transferred** to other venues while retaining their original dates:

- 23 July → Southwell;
- 6 August → Southwell;
- 13 August → Lingfield Park.

The BHA continued to describe them as the three fixtures and stated that their race programmes would largely remain the same, subject to course-specific alterations.

This demonstrates that the originally scheduled **racecourse is not inherently the identity of the fixture**.

A fixture can survive a venue transfer.

Locator:

https://www.britishhorseracing.com/press_releases/bha-confirms-the-transfer-of-three-chelmsford-city-fixtures/


### Evidence 3 — A fixture can change date

**Source:** British Horseracing Authority, *Ffos Las, Kempton Park and Salisbury fixtures rescheduled; Newmarket and Nottingham to start earlier in the day*  
**Published:** 22 June 2026

Fixtures scheduled for Wednesday 24 June were postponed because of extreme heat.

The BHA subsequently rescheduled:

- Ffos Las → afternoon of Monday 29 June;
- Kempton Park → evening of Monday 29 June;
- Salisbury → afternoon of Tuesday 30 June.

The BHA stated that the race programmes and prize money at all three fixtures would remain the same as the originally scheduled cards.

The terminology therefore treats these as **rescheduled fixtures**, rather than simply unrelated new fixtures created because the date changed.

This demonstrates that the originally scheduled **calendar date is not inherently the identity of the fixture**.

Locator:

https://www.britishhorseracing.com/press_releases/ffos-las-kempton-park-and-salisbury-fixtures-rescheduled-newmarket-and-nottingham-to-start-earlier-in-the-day/


### Evidence 4 — Race times can change without changing the fixture

**Source:** British Horseracing Authority, *BHA confirms later start time for Windsor's fixture on Saturday 27 June*  
**Published:** 24 June 2026

The first race at Windsor was moved from 13:50 to 17:15 and the final race from 17:10 to 20:15 because of forecast high temperatures.

Throughout the announcement the BHA continued to refer to the event as Windsor's fixture on Saturday 27 June.

Therefore an exact first-race time, last-race time or clock-time range cannot itself constitute fixture identity.

Locator:

https://www.britishhorseracing.com/press_releases/bha-confirms-later-start-time-for-windsors-fixture-on-saturday-27-june/


### Evidence 5 — Session is an explicit feature of the Fixture List

**Source:** British Horseracing Authority, *BHA publishes 2026 fixture list*  
**Published:** 6 August 2025

The BHA reports the annual Fixture List not only by racing code but also by **session**.

The published 2026 Fixture List distinguishes fixtures using labels including:

- `(E)` — Evening;
- `(F)` — Floodlit.

The BHA therefore treats session as a meaningful characteristic of fixtures and of the overall Fixture List.

This is particularly relevant given the historical Wolverhampton double-header evidence, where an afternoon fixture and an evening fixture at the same racecourse on the same date remained separate fixtures.

However, this evidence does **not yet establish that session is itself a permanent fixture identifier**. It establishes only that session is an administratively meaningful fixture attribute.

Locators:

https://www.britishhorseracing.com/press_releases/bha-publishes-2026-fixture-list/

https://media.britishhorseracing.com/bha/Fixture_List/2026_Fixture_List.pdf


### Evidence status

The boundary cases establish that a fixture is not simply equivalent to any one of:

- racecourse;
- date;
- exact race times;
- exact race programme.

Those properties can change while BHA terminology continues to track a fixture through the change.

The evidence also establishes that **session** is a real Fixture List concept and that, historically, different sessions have distinguished separate fixtures at the same racecourse on the same date.

What remains unresolved is the precise administrative identity carried by a fixture underneath those mutable properties.

## Question 3 — Are all fixtures the same kind of administrative object?

### Why this follows from Question 2

Question 2 showed that a fixture cannot safely be defined by its current racecourse, date or race times alone.

The BHA's race-planning material introduces another distinction: British fixtures themselves are not all allocated in the same way.

Before attempting to model fixture identity, we therefore need to understand the BHA's own fixture categories.

### Method

Use BHA race-planning material to establish:

- what a **Racecourse Fixture** is;
- what a **BHA Fixture** is;
- how each can be allocated or moved;
- whether these categories further support treating a fixture as an administrative object rather than merely a group of races at a course on a date.

## Question 3 — Evidence notes

### Evidence 1 — The BHA distinguishes two types of fixture

**Source:** British Horseracing Authority, *Race Planning*

The BHA states that, within the framework used to compile the Fixture List, there are two types of fixture:

1. **Racecourse Fixtures**
2. **BHA Fixtures**

This is an explicit administrative distinction within the fixture system.

Locator:

https://www.britishhorseracing.com/about/planning/


### Evidence 2 — Racecourse Fixtures can be moved or swapped

For **Racecourse Fixtures**, the BHA states that racecourses may request to swap and move fixtures with other courses.

The BHA Racing Department oversees these transfers to ensure that the wider interests of the Fixture List are maintained.

This reinforces the earlier finding that a fixture is not simply synonymous with the racecourse currently staging it.


### Evidence 3 — BHA Fixtures are allocated through a bidding process

For **BHA Fixtures**, the BHA describes a separate allocation system.

A number of BHA Fixtures are offered on a **leasehold basis**, with racecourses bidding competitively to stage them.

The successful racecourse is granted the fixture subject to regulatory approval.

This provides particularly strong evidence that, at least for BHA Fixtures, the fixture concept exists separately from the identity of the racecourse that ultimately stages it.


### Evidence 4 — Fixture planning and race programming are related but distinct activities

The BHA states that its Racing Department is responsible for compiling:

- the **Fixture List**; and
- all **race programmes**.

The same material discusses the ability to transfer fixtures or major races at short notice.

This suggests that the scheduled fixture and the races programmed within it are related but conceptually distinct objects.

Locator:

https://www.britishhorseracing.com/about/planning/


### What this establishes

British racing has at least two administrative fixture categories:

| Fixture type | Administrative feature |
|---|---|
| Racecourse Fixture | Associated with racecourse fixture rights and capable of approved movement or swapping |
| BHA Fixture | Allocated by the BHA through a bidding/leasehold process |

The distinction strengthens the interpretation developed in Question 2:

> A fixture is an administrative scheduling object to which a racecourse, date and race programme are assigned; it is not merely the accidental combination of those observable properties.

### What this does not establish

This does not yet tell us:

- whether Racecourse Fixtures and BHA Fixtures are distinguishable in ordinary published race results;
- whether the category remains relevant once the fixture has actually been run;
- how individual races relate to the fixture administratively;
- whether a race can be moved independently from one fixture to another;
- whether Inside Rails needs to represent fixture ownership/allocation type.

## Question 4 — What is the relationship between a fixture, its race programme and its races?

### Why this matters

A fixture contains racing, but that does not necessarily mean that the particular set of races initially programmed for it defines the fixture.

We need to distinguish:

- the fixture itself;
- the programme of races assigned to it;
- the individual races;
- ordinary use of the word **card**.

This matters because, if races can be added, removed or transferred while the fixture continues to exist, the race programme cannot itself be the fixture's identity.

### Method

Use BHA examples in which race programmes or individual races change while the surrounding fixture remains identifiable.

## Question 4 — Evidence notes

### Evidence 1 — The BHA distinguishes the Fixture List from the race programme

**Source:** British Horseracing Authority, *Race Planning*

The BHA states that its Racing Department is responsible for compiling both:

- the **Fixture List**; and
- all **race programmes**.

It describes the race programme as the programme of races designed to provide appropriate opportunities for the horse population across ability levels, distances and the season.

The BHA therefore treats fixture planning and race programming as related but distinct activities.

Locator:

https://www.britishhorseracing.com/about/planning/


### Evidence 2 — Races can be removed and replaced without replacing the fixture

**Source:** British Horseracing Authority, *BHA confirms changes to Redcar's race programme on Monday 15 April*  
**Published:** 8 April 2024

Because part of Redcar's course was waterlogged, three scheduled 10-furlong races were abandoned.

The BHA then programmed three different races in order to give **the fixture** the best chance of going ahead.

The fixture therefore persisted while several of the races assigned to it changed.

Locator:

https://www.britishhorseracing.com/press_releases/bha-confirms-changes-to-redcars-race-programme-on-monday-15-april/


### Evidence 3 — An individual race can move from one fixture to another

**Source:** British Horseracing Authority, *Additional fixtures scheduled at Wolverhampton on 29 May and Carlisle on 30 May*  
**Published:** 26 May 2026

Following cancelled Haydock fixtures, the BHA scheduled an additional fixture at Carlisle.

The Carlisle card included the **Silver Bowl**, which had been rescheduled from a Haydock fixture.

The race therefore retained enough continuity to be described as the Silver Bowl despite moving from one fixture to another.

This demonstrates that an individual race and the fixture containing it are distinct concepts.

Locator:

https://www.britishhorseracing.com/press_releases/additional-fixtures-scheduled-at-wolverhampton-on-29-may-and-carlisle-on-30-may/


### Evidence 4 — A transferred fixture can receive an altered race programme

**Source:** British Horseracing Authority, *Musselburgh to host replacement Sunday Series fixture on 26 April*  
**Published:** 20 April 2026

A Sunday Series fixture originally programmed for Ayr was replaced at Musselburgh.

The BHA stated that the new venue required an **updated race programme** because of differences in track configuration and starting positions.

Several race distances therefore changed while the wider fixture concept and Sunday Series role were retained.

Locator:

https://www.britishhorseracing.com/press_releases/musselburgh-to-host-replacement-sunday-series-fixture-on-26-april/


### What this establishes

The evidence supports the following structure:

**fixture**
→ has a **race programme**
→ containing individual **races**

But these are not identical objects.

A fixture can survive changes to its race programme.

An individual race can also be transferred from one fixture to another.

The set of races on a card therefore cannot safely be used as the permanent identity of the fixture.

### What about "card"?

The BHA regularly uses **card** as ordinary shorthand for the collection of races scheduled at a fixture — for example, "six-race card" or "originally scheduled card".

Nothing found so far requires Inside Rails to model **card** as a separate administrative entity between fixture and race programme.

For the purposes of this study, **card** can therefore remain reader-facing shorthand unless later evidence establishes a distinct technical meaning.

## Question 5 — How does the BHA identify completed fixtures to users?

### Why this matters

Previous questions established that a fixture has administrative continuity even when attributes such as venue, date, race times or race programme change.

We now need to distinguish that underlying administrative identity from the information by which a completed fixture is presented to the public.

The question is therefore:

> What information does the BHA results service use to describe and distinguish fixtures once racing has taken place?

### Method

Inspect the current BHA results service and record the fixture-level characteristics exposed before opening the individual races.

## Question 5 — How does the BHA identify completed fixtures to users?

### Why this matters

Previous questions established that a fixture has administrative continuity even when attributes such as venue, date, race times or race programme change.

We now need to distinguish that underlying administrative identity from the information by which a completed fixture is presented to the public.

The question is therefore:

> What information does the BHA results service use to describe and distinguish fixtures once racing has taken place?

### Method

Inspect the current BHA results service and record the fixture-level characteristics exposed before opening the individual races.

## Question 5 — Evidence notes

### Evidence 1 — The BHA results service searches for fixtures, not merely individual races

**Source:** British Horseracing Authority, *Results*

The BHA results page describes its purpose as finding racing **fixtures** in the UK.

Its fixture-level search and display structure exposes:

- date;
- fixture type;
- racecourse;
- first-race time;
- fixture time/session.

Fixture type is represented as:

- Flat;
- Jump;
- Mixed.

Fixture time/session is represented as:

- Afternoon;
- Twilight;
- Evening.

The result template itself uses BHA fields named:

- `fixtureDate`;
- `fixtureType`;
- `courseName`;
- `firstRace`;
- `fixtureSession`.

A user selects the fixture and then chooses **View results** to inspect the races within it.

Locator:

https://www.britishhorseracing.com/racing/results/


### Evidence 2 — Upcoming fixtures use substantially the same fixture-level structure

**Source:** British Horseracing Authority, *Upcoming fixtures*

The BHA's upcoming-fixture service similarly presents a scheduled fixture using:

- fixture date;
- fixture type;
- racecourse;
- first-race time;
- fixture session.

From that fixture the user can then access entries, declarations tracking and the racecard.

This supports the structural distinction already established:

**fixture**
→ fixture-level attributes
→ races/racecard/results

Locator:

https://www.britishhorseracing.com/racing/fixtures/upcoming/


### Interpretation

The BHA public services expose enough fixture-level attributes for a human reader to recognise and select a fixture.

However, these attributes should not be mistaken for a permanent fixture identity.

Earlier evidence has already shown that:

- racecourse can change;
- date can change;
- first-race time can change;
- session can change with rescheduling;
- the race programme can change.

The public-facing BHA results page therefore provides a **description of the realised fixture**, not evidence that any particular combination of those displayed values is the fixture's durable administrative key.

### Public identifier boundary

No persistent fixture number or fixture ID is visibly presented to the user in the current results listing.

This does not establish that the BHA has no internal fixture identifier.

It establishes only that the public results interface does not require readers to know one: fixtures are presented through their observable characteristics.

## Question 5 — What we found

The BHA distinguishes a fixture publicly using a bundle of realised scheduling attributes rather than exposing an obvious persistent fixture identifier.

For analytical reconstruction, this creates an important distinction:

> **A set of attributes may be sufficient to distinguish observed fixtures in a dataset without constituting the true administrative identity of the fixture.**

Inside Rails therefore does not need to claim that it has reconstructed the BHA's hidden administrative fixture identity unless the evidence genuinely supports that claim.

The next task is to determine what Database v4 actually contains and what level of fixture reconstruction those fields can support.

## Question 6 — What does the source look like under the provisional racecourse-date grouping?

### Existing database boundary

The database design already establishes that races sharing a date and course/racecourse are only **candidate members** of one meeting.

Earlier database design explicitly rejected `date + course` as a permanent meeting identity because a same-date, same-venue group could conceal:

- separate cards or sessions;
- transferred or rearranged racing;
- split or resumed racing;
- other exceptional structures.

Study 04 therefore does not need to rediscover the database schema.

The current task is narrower:

> What does the Great Britain race population actually look like when grouped by governed racecourse and source date, without claiming those groups are fixtures?

This provides a candidate population in which possible fixture-boundary exceptions can be investigated.

In [3]:
query = """
WITH racecourse_dates AS (
    SELECT
        CAST(raw_date AS TEXT) AS raw_date,
        racecourse_identity_code,
        governed_racecourse_name,
        COUNT(*) AS races,
        COUNT(DISTINCT raw_course) AS source_course_labels
    FROM view_gb_reconciled_race_occurrences_with_racecourse
    GROUP BY
        CAST(raw_date AS TEXT),
        racecourse_identity_code,
        governed_racecourse_name
)
SELECT
    races,
    COUNT(*) AS racecourse_date_groups
FROM racecourse_dates
GROUP BY races
ORDER BY races
"""

largest_query = """
SELECT
    CAST(raw_date AS TEXT) AS raw_date,
    racecourse_identity_code,
    governed_racecourse_name,
    COUNT(*) AS races,
    COUNT(DISTINCT raw_course) AS source_course_labels,
    GROUP_CONCAT(DISTINCT CAST(raw_course AS TEXT)) AS raw_course_labels
FROM view_gb_reconciled_race_occurrences_with_racecourse
GROUP BY
    CAST(raw_date AS TEXT),
    racecourse_identity_code,
    governed_racecourse_name
ORDER BY races DESC, raw_date, governed_racecourse_name
LIMIT 30
"""

with connect_read_only(DATABASE) as connection:
    group_size_distribution = pd.read_sql_query(query, connection)
    largest_racecourse_dates = pd.read_sql_query(largest_query, connection)

display(group_size_distribution)
display(largest_racecourse_dates)

,races,racecourse_date_groups
0,1,5
1,2,12
2,3,14
3,4,18
4,5,19
5,6,2871
6,7,8879
7,8,3373
8,9,537
9,10,20


,raw_date,racecourse_identity_code,governed_racecourse_name,races,source_course_labels,raw_course_labels
0,2020-06-01,rc:gb:newcastle,Newcastle,10,1,Newcastle (AW)
1,2020-06-02,rc:gb:newcastle,Newcastle,10,1,Newcastle (AW)
2,2020-06-03,rc:gb:great-yarmouth,Great Yarmouth,10,1,Yarmouth
3,2020-06-04,rc:gb:newcastle,Newcastle,10,1,Newcastle (AW)
4,2020-06-06,rc:gb:newcastle,Newcastle,10,1,Newcastle (AW)
5,2020-06-08,rc:gb:chelmsford-city,Chelmsford City,10,1,Chelmsford (AW)
6,2020-06-16,rc:gb:chelmsford-city,Chelmsford City,10,1,Chelmsford (AW)
7,2020-06-26,rc:gb:doncaster,Doncaster,10,1,Doncaster
8,2020-06-30,rc:gb:doncaster,Doncaster,10,1,Doncaster
9,2020-07-09,rc:gb:york,York,10,1,York


### What this establishes

The largest governed racecourse-date groups in Database v4 contain 10 race occurrences.

Large race counts do not themselves establish multiple fixtures.

The observed population therefore provides no defensible race-count threshold at which a racecourse-date group should automatically be split.

The next step is to look for **internal temporal structure** within these candidate groups.

A large gap between races will be treated only as candidate evidence for further investigation, not as proof of a fixture or session boundary.

## Question 7 — Do any racecourse-date groups show possible internal card boundaries?

### Why this matters

British racing can, at least in principle, stage more than one fixture at the same racecourse on the same date.

Database v4 does not contain a governed fixture assignment.

We can therefore use race timing only to identify **candidates for external verification**.

The aim is not to define a session by an arbitrary time-gap rule.

Instead:

> rank racecourse-date groups by their largest observed gap between consecutive resolved advertised start times, then inspect the strongest candidates individually.

In [4]:
query = """
WITH ordered_races AS (
    SELECT
        r.raw_date,
        r.racecourse_identity_code,
        r.governed_racecourse_name,
        r.source_race_occurrence_code,
        t.advertised_start_uk,
        LAG(t.advertised_start_uk) OVER (
            PARTITION BY
                r.raw_date,
                r.racecourse_identity_code
            ORDER BY t.advertised_start_uk
        ) AS previous_start
    FROM view_gb_reconciled_race_occurrences_with_racecourse AS r
    JOIN core_source_race_occurrence_time AS t
      ON t.source_race_occurrence_id = r.source_race_occurrence_id
    WHERE t.temporal_resolution_status = 'resolved'
),
gaps AS (
    SELECT
        *,
        ROUND(
            (julianday(advertised_start_uk) - julianday(previous_start))
            * 24 * 60,
            1
        ) AS gap_minutes
    FROM ordered_races
),
group_resolution AS (
    SELECT
        r.raw_date,
        r.racecourse_identity_code,
        COUNT(*) AS total_races,
        SUM(
            CASE
                WHEN t.temporal_resolution_status = 'resolved' THEN 1
                ELSE 0
            END
        ) AS resolved_races
    FROM view_gb_reconciled_race_occurrences_with_racecourse AS r
    JOIN core_source_race_occurrence_time AS t
      ON t.source_race_occurrence_id = r.source_race_occurrence_id
    GROUP BY
        r.raw_date,
        r.racecourse_identity_code
)
SELECT
    g.raw_date,
    g.racecourse_identity_code,
    g.governed_racecourse_name,
    gr.total_races AS races,
    MAX(g.gap_minutes) AS largest_gap_minutes
FROM gaps AS g
JOIN group_resolution AS gr
  ON gr.raw_date = g.raw_date
 AND gr.racecourse_identity_code = g.racecourse_identity_code
WHERE gr.total_races = gr.resolved_races
GROUP BY
    g.raw_date,
    g.racecourse_identity_code,
    g.governed_racecourse_name,
    gr.total_races
HAVING gr.total_races >= 2
ORDER BY
    largest_gap_minutes DESC,
    races DESC
LIMIT 30
"""

with connect_read_only(DATABASE) as connection:
    largest_internal_gaps = pd.read_sql_query(query, connection)

display(largest_internal_gaps)

,raw_date,racecourse_identity_code,governed_racecourse_name,races,largest_gap_minutes
0,2019-10-17,rc:gb:brighton,Brighton,3,165.0
1,2022-05-30,rc:gb:lingfield-park,Lingfield Park,5,100.0
2,2022-11-01,rc:gb:redcar,Redcar,6,95.0
3,2018-03-30,rc:gb:bath,Bath,4,95.0
4,2025-08-29,rc:gb:thirsk,Thirsk,5,90.0
5,2024-03-13,rc:gb:cheltenham,Cheltenham,6,80.0
6,2023-06-20,rc:gb:stratford-on-avon,Stratford-on-Avon,5,80.0
7,2019-06-15,rc:gb:bath,Bath,5,75.0
8,2026-05-23,rc:gb:haydock-park,Haydock Park,4,73.0
9,2016-09-08,rc:gb:doncaster,Doncaster,7,70.0


## Question 7 — Evidence notes

The racecourse-date groups with the largest internal gaps were checked against external race-day evidence.

### Brighton — 17 October 2019

Database v4 contains three completed races with a maximum internal gap of 165 minutes.

Brighton Racecourse explains why.

The first two races were run, after which unsafe false ground caused races 3–6 to be abandoned. The seventh and final race used a different part of the course and was therefore still able to take place at its originally scheduled time of 16:40.

The resulting long gap was therefore created by **four abandoned races within one card**, not by a boundary between two fixtures.

Brighton itself describes the occasion as its final **meeting** of the 2019 season.

Sources:

https://www.brighton-racecourse.co.uk/news/racing/race-report-17th-october

https://www.brighton-racecourse.co.uk/news/racing/brighton-racecourse-statement-17-october


### Lingfield Park — 30 May 2022

Database v4 contains five completed races with a maximum internal gap of 100 minutes.

Contemporary results record that races 2 and 3 were abandoned because of unsafe ground after horses slipped during the opening race.

Racing subsequently continued with the remaining races.

The gap therefore represents **missing races from a partially abandoned card**, not evidence of two fixtures.

Source:

https://www.racingpost.com/results/2022-05-30


### Redcar — 1 November 2022

Database v4 contains six completed races with a maximum internal gap of 95 minutes.

Contemporary results record that the scheduled 15:05 and 15:35 races were abandoned because of standing water.

Again, the large temporal gap is explained by abandoned races within the day's programme rather than a second fixture.

Source:

https://www.racingpost.com/results/2022-11-01


### Bath — 30 March 2018

Database v4 contains four completed races with a maximum internal gap of 95 minutes.

Bath Racecourse states that the round course became unraceable.

Three races were abandoned while four races using the straight courses remained scheduled at:

- 13:50;
- 14:20;
- 15:30;
- 17:05.

The apparently fragmented sequence therefore belongs to a curtailed day's racing rather than establishing separate fixtures.

Source:

https://www.bath-racecourse.co.uk/news/racing/statement


### Further high-gap examples

The same phenomenon appears elsewhere among the strongest candidates:

- Thirsk, 29 August 2025 — two later races abandoned because of safety concerns;
- Cheltenham, 13 March 2024 — the Cross Country Chase was abandoned;
- Stratford-on-Avon, 20 June 2023 — one race was voided following a jockey fall;
- Bath, 15 June 2019 — round-course races were abandoned while straight-course racing continued.

These cases reinforce the same analytical warning.

## Question 7 — What we found

Large gaps between consecutive completed races are **not reliable fixture or session boundaries**.

In the strongest observed candidates, large gaps frequently arise because scheduled races were:

- abandoned;
- voided;
- or otherwise absent from the completed-race source.

This exposes an important limitation of reconstructing meeting structure from a results-led dataset:

> absence of a race occurrence does not imply absence of a scheduled race.

Consequently, neither:

`racecourse + date`

nor:

`racecourse + date + temporal gap`

is sufficient on its own to establish fixture identity.

Temporal structure remains useful as an **anomaly-detection signal**, but any apparent split requires independent evidence.

## Question 8 — Does Database v4 contain evidence of two completed fixtures at the same racecourse on the same date?

### Why this follows from Question 7

Question 7 showed that apparent internal splits can be false positives caused by abandonment.

We now need to look for positive evidence of the opposite structure:

> two genuinely distinct fixtures staged at the same governed racecourse on the same date.

The BHA has historically permitted such double-header arrangements, so the structure is possible in British racing.

However, possibility does not establish that one occurs in the Inside Rails period from 2015 onward.

### Method

Use the observed racecourse-date population together with BHA fixture evidence.

Do not infer two fixtures merely from race counts or race-time gaps.

Instead, investigate whether any racecourse-date in the source period can be independently verified as containing separately scheduled fixtures.

## Question 8 — Evidence notes

### Evidence 1 — No racecourse-date group contains enough completed races to expose an obvious fully completed double-header

Across the 111,634 Great Britain race occurrences in Database v4, the largest observed governed racecourse-date group contains:

**10 completed races**

Earlier BHA evidence established that a fixture is normally programmed with at least six races.

Therefore, if two ordinary six-race-or-larger fixtures were both completed in full at the same racecourse on the same date, the results-led source would ordinarily expose at least 12 completed race occurrences.

No such racecourse-date group exists in Database v4.

This does not prove that no double-header occurred, because abandoned or otherwise uncompleted races may be absent from the source.


### Evidence 2 — The 10-race groups are concentrated in the exceptional 2020 resumption period

Most of the largest racecourse-date groups occur during the summer of 2020.

BHA resumption planning explicitly anticipated extending fixtures beyond the normal number of races because the post-lockdown fixture list contained fewer fixtures and racing opportunities needed to be preserved.

The BHA subsequently referred directly to operational arrangements for **10-race cards**.

The observed 10-race groups are therefore consistent with unusually large individual fixtures during the COVID-19 resumption period rather than providing evidence of two fixtures being merged in the source.

Sources:

https://www.britishhorseracing.com/press_releases/update-on-resumption-of-racing-planning/

https://www.britishhorseracing.com/press_releases/resumption-update-race-programme-and-planning/


### Evidence 3 — Same-course double-headers are structurally possible

The historical 2012 Wolverhampton evidence remains important.

The BHA explicitly arranged an afternoon fixture at Wolverhampton with a gap before Wolverhampton's existing evening fixture and described the result as a **double header**.

Therefore British fixture administration does permit:

> same racecourse + same date + more than one fixture

However, that verified example predates the Inside Rails source period.


### Evidence 4 — No equivalent case has yet been verified within the Inside Rails period

A targeted search of BHA fixture-change and additional-fixture material from the Inside Rails period found many examples of:

- additional fixtures;
- transferred fixtures;
- rescheduled fixtures;
- afternoon, twilight and evening fixtures.

However, no post-2015 case has yet been verified in which the BHA explicitly identifies two separate horse-racing fixtures staged at the same racecourse on the same date.

This is an evidence boundary, not proof that no such case occurred.

## Question 8 — What we found

Database v4 contains **no positive evidence of two completed fixtures at the same governed racecourse on the same source date**.

The maximum observed group is 10 completed races, and the largest groups are substantially explained by documented extended-card arrangements during the exceptional 2020 resumption period.

However:

> absence of an observed double-header does not make `date + racecourse` the definition of a fixture.

The BHA has demonstrably permitted multiple fixtures at one racecourse on one date, and a results-led dataset can conceal scheduled races that were abandoned or otherwise never produced results.

The defensible conclusion is therefore:

> `source date + governed racecourse` is a strong **candidate grouping for realised racing**, but it is not a proven BHA fixture identity.

Any claim that such a group represents one official fixture requires either independent fixture evidence or an explicitly stated approximation.

## Question 9 — Can Inside Rails reconstruct administrative fixture identity from completed race results alone?

### Why this follows from Question 8

A racecourse-date grouping describes where and when completed races appear in the source.

But earlier BHA evidence established that an administrative fixture can survive:

- a venue transfer;
- a date change;
- substantial changes to its race programme;
- substantial changes to race times.

The question is therefore no longer simply how to split a racecourse-date group.

It is:

> Does a completed-results source contain enough information to recover the underlying BHA fixture identity at all?

### Method

Use verified transferred and rescheduled fixture examples to compare:

1. the administrative history described by the BHA; with
2. the realised race occurrence that a results-led database can observe.

Determine which fixture facts are lost if only completed race results are retained.

## Question 9 — Administrative fixture continuity versus realised race results

### Evidence 1 — Brighton fixtures transferred to other racecourses

**Source:** British Horseracing Authority, *BHA confirms the transfer of Brighton's next three fixtures*  
**Published:** 20 April 2026

Following problems with the Brighton track, the BHA described three already scheduled **Brighton fixtures** as being moved to other racecourses:

- Tuesday 28 April → Yarmouth;
- Wednesday 29 April → Bath;
- Thursday 7 May → Windsor.

The BHA stated that the race programmes at the transferred fixtures would mirror the originally scheduled cards as closely as possible.

The important identity point is that the BHA describes these administratively as Brighton's fixtures even though they are subsequently staged at different racecourses. :contentReference[oaicite:0]{index=0}

Locator:

`https://www.britishhorseracing.com/press_releases/bha-confirms-the-transfer-of-brightons-next-three-fixtures/`


### Evidence 2 — Chepstow fixture transferred to Bath

**Source:** British Horseracing Authority, *BHA confirms transfer of Chepstow's fixture on Tuesday 12 May to Bath*  
**Published:** 3 February 2026

The BHA states that **Chepstow's fixture on Tuesday 12 May** was moved to Bath because of drainage works at Chepstow.

Again, the fixture has an administrative origin associated with Chepstow while its realised racing takes place at Bath. :contentReference[oaicite:1]{index=1}

Locator:

`https://www.britishhorseracing.com/press_releases/bha-confirms-transfer-of-chepstows-fixture-on-tuesday-12-may-to-bath/`


### Why these cases matter

All four transferred fixtures fall inside the Inside Rails source period.

They therefore provide direct tests of what Source Version 1 and Database v4 retain after a transferred fixture has actually been staged.

If the race occurrences record only the realised date and staging racecourse, then administrative fixture history cannot be reconstructed from completed race results alone.

In [7]:
# These are the four 2026 fixtures for which BHA evidence establishes
# an administrative transfer from the originally scheduled racecourse
# to a different staging racecourse.
#
# Brighton:
#   28 April -> Great Yarmouth
#   29 April -> Bath
#    7 May   -> Windsor
#
# Chepstow:
#   12 May   -> Bath
#
# The purpose of this query is NOT to reconstruct fixture identity.
# It is to test what Database v4 retains about the realised racing
# after those administratively transferred fixtures were staged.

transfer_dates = (
    "2026-04-28",
    "2026-04-29",
    "2026-05-07",
    "2026-05-12",
)

placeholders = ", ".join("?" for _ in transfer_dates)

query = f"""
SELECT
    CAST(raw_date AS TEXT) AS race_date,
    governed_racecourse_name,
    racecourse_identity_code,
    COUNT(*) AS completed_races
FROM view_gb_reconciled_race_occurrences_with_racecourse

-- Restrict the population to the independently verified transfer dates.
WHERE CAST(raw_date AS TEXT) IN ({placeholders})

-- Limit the possible venues to the original and realised racecourses
-- involved in those transfer cases.
  AND governed_racecourse_name IN (
      'Brighton',
      'Great Yarmouth',
      'Bath',
      'Windsor',
      'Chepstow'
  )

-- Summarise the completed race occurrences at the realised
-- racecourse-date level. This does not claim that the grouping
-- is itself an official fixture identity.
GROUP BY
    CAST(raw_date AS TEXT),
    governed_racecourse_name,
    racecourse_identity_code

ORDER BY
    race_date,
    governed_racecourse_name
"""

# Database v4 remains immutable and is opened through the governed
# project read-only connection helper.
with connect_read_only(DATABASE) as connection:
    transferred_fixture_observations = pd.read_sql_query(
        query,
        connection,
        params=transfer_dates,
    )

display(transferred_fixture_observations)

,race_date,governed_racecourse_name,racecourse_identity_code,completed_races
0,2026-04-28,Great Yarmouth,rc:gb:great-yarmouth,7
1,2026-04-29,Bath,rc:gb:bath,8
2,2026-05-07,Windsor,rc:gb:windsor,6
3,2026-05-12,Bath,rc:gb:bath,6


## Question 9 — What we found

The transferred-fixture test establishes a hard information boundary between BHA fixture administration and the completed-race evidence available in Database v4.

The BHA identifies:

- the fixtures on 28 April, 29 April and 7 May 2026 as fixtures originally scheduled for **Brighton**, subsequently transferred to Great Yarmouth, Bath and Windsor;
- the fixture on 12 May 2026 as **Chepstow's fixture**, subsequently transferred to Bath.

Database v4, by contrast, observes the realised racing as:

| Date | Governed racecourse | Completed races |
|---|---|---:|
| 28 April 2026 | Great Yarmouth | 7 |
| 29 April 2026 | Bath | 8 |
| 7 May 2026 | Windsor | 6 |
| 12 May 2026 | Bath | 6 |

The realised race occurrences therefore preserve **where the races were staged**, but not the administrative history that identifies the first three as transferred Brighton fixtures or the fourth as a transferred Chepstow fixture.

### Conclusion

> **Administrative BHA fixture identity cannot be reconstructed reliably from completed race results alone.**

A results-led source can describe the realised racing occurrence:

- when racing took place;
- at which governed racecourse;
- which races produced results;
- and other race-level characteristics.

It does not by itself preserve all of the administrative facts required to identify the underlying scheduled fixture, including its original venue or transfer history.

This distinction also explains why deriving a fixture merely from `date + racecourse` would be conceptually wrong.

For example, the races staged at Great Yarmouth on 28 April 2026 form an observable Great Yarmouth racecourse-date group in Database v4, while BHA administrative evidence establishes that the underlying scheduled fixture originated at Brighton.

### Modelling boundary

Inside Rails can reconstruct a **realised source-level grouping of races** from its results evidence.

It cannot claim that such a grouping is the BHA's persistent administrative fixture identity unless separate fixture evidence establishes that relationship.

A governed administrative fixture layer would therefore require an additional fixture-level evidence source rather than being inferred solely from completed race occurrences.

## Question 10 — What should Inside Rails call the realised grouping observable in the source?

### Why this follows from Question 9

Study 04 has established two different concepts.

The BHA has an administrative **fixture** that can retain continuity through changes to:

- venue;
- date;
- race times;
- and race programme.

Source Version 1, however, is primarily completed-race evidence.

It can support grouping races that were realised together at a governed racecourse on a source date, but it does not contain enough information to claim that this grouping is the BHA's persistent administrative fixture identity.

We therefore need a name for the source-level entity that:

1. accurately describes what the evidence supports;
2. does not falsely claim BHA administrative fixture identity;
3. remains useful for meeting-level analytical questions;
4. preserves exceptional or unresolved structures rather than forcing them into a false key.

### Existing design context

The earlier Phase 3 design proposed the term:

**source meeting occurrence**

with explicit governed race-to-meeting assignments.

That design deliberately treated `raw date + raw course` only as candidate grouping evidence and deferred the actual meeting-structure rules to a dedicated study.

Study 04 can now test whether that terminology remains appropriate in light of the authoritative BHA distinction between **meeting** and **fixture** established above.

## Question 10 — What should Inside Rails call the realised grouping observable in the source?

### Evidence — "raceday" is not a more precise substitute

BHA usage does not provide a third technical term that solves the problem.

For example, in its July 2025 announcement about Redcar, the BHA describes the same event as:

- a **raceday**;
- a **fixture**;
- and a **meeting**.

This shows that *raceday*, like *meeting*, is useful ordinary racing language but does not provide a more precise persistent administrative identity than *fixture*.

**Source:** British Horseracing Authority, *New apprentice-only raceday to take place at Redcar on Sunday*  
**Published:** 18 July 2025  
https://www.britishhorseracing.com/press_releases/new-apprentice-only-raceday-to-take-place-at-redcar-on-sunday/

The BHA also uses *raceday* operationally for what happens at or around a fixture, while continuing to use *fixture* for the scheduled administrative object.

**Source:** British Horseracing Authority, *New BHA policy to permit race surface switch on raceday*  
**Published:** 7 May 2024  
https://www.britishhorseracing.com/press_releases/new-bha-policy-to-permit-race-surface-switch-on-raceday/

### Existing Inside Rails terminology

The earlier Phase 3 design proposed:

**source meeting occurrence**

That was deliberately provisional and already warned that `date + course` did not prove meeting identity.

Study 04 has now supplied the missing domain evidence.

The term **source meeting occurrence** is too strong for the grouping that Database v4 can reconstruct because:

- BHA *meeting* can mean a single day's racing or a wider multi-day event;
- the source cannot preserve BHA administrative fixture continuity through transfers;
- same-racecourse, same-date racing is not guaranteed to represent exactly one fixture;
- partial abandonments can make the completed race sequence incomplete.

### Revised terminology

For the structure observable directly in Database v4, use:

> **source racecourse-date group**

Definition:

> The set of observed Great Britain race occurrences sharing one source date and one governed racecourse identity.

This is deliberately a descriptive analytical grouping rather than a claimed racing entity.

It does not assert that the group is:

- one BHA fixture;
- one BHA meeting;
- one raceday in every possible racing sense;
- one complete race programme;
- or one persistent administrative object.

### Conclusion

The terminology should therefore remain separated:

| Term | Inside Rails treatment |
|---|---|
| **fixture** | BHA administrative scheduling object; not reconstructable from Database v4 alone |
| **meeting** | contextual racing term; may refer to one fixture/day or a wider multi-day event |
| **raceday** | useful ordinary/operational term, but not adopted as a technical identity |
| **card / race programme** | races associated with a fixture; not fixture identity |
| **source racecourse-date group** | observable analytical grouping supported directly by Database v4 |

The earlier conceptual term **source meeting occurrence** should therefore not be implemented unchanged.

## Question 11 — What entity structure does Study 04 actually justify?

### Why this matters

Study 04 has now separated three things that earlier design work could not yet distinguish confidently:

1. the BHA's administrative **fixture**;
2. contextual racing terms such as **meeting** and **raceday**;
3. the realised grouping of completed races observable in Database v4.

The remaining question is therefore practical:

> What should Inside Rails actually represent from the evidence it currently possesses?

### Evidence available to Inside Rails

Database v4 already contains:

- one governed Great Britain race occurrence per row;
- source date;
- governed racecourse identity;
- governed race-level classifications and other race attributes;
- governed advertised start-time reconstruction where timing is required.

From these facts, races can be grouped analytically by:

`source date + governed racecourse identity`

Study 04 has named this a:

**source racecourse-date group**

However, the study has also established that this grouping cannot safely be promoted to a BHA fixture identity because:

- two fixtures can in principle occur at one racecourse on one date;
- large gaps between completed races can be caused by abandonment rather than separate fixtures;
- a fixture can move to another racecourse;
- a fixture can move to another date;
- its race programme and race times can change;
- completed results do not preserve the fixture's full administrative history.

### Smallest correct model

Study 04 therefore justifies the following structure:

```text
BHA administrative fixture
        |
        |  not reconstructable from Database v4 alone
        |
        |  requires separate fixture-level evidence
        v

realised races in Source Version 1
        |
        v
source racecourse-date group

## Question 12 — What authoritative evidence would be needed for a genuine fixture layer?

### Why this follows from Question 11

Study 04 has established that Database v4 can describe realised racing but cannot reconstruct persistent BHA fixture identity from completed race results alone.

That does not mean fixture identity is unknowable.

The next question is:

> What BHA evidence would Inside Rails need if it later wanted to build a governed administrative fixture layer?

### Evidence 1 — The BHA publishes an official annual Fixture List

**Source:** British Horseracing Authority, *Full Year*

The BHA states that the Fixture List is compiled by its Racing Department.

For 2026 it provides the official Fixture List in both:

- Excel;
- PDF.

The BHA's 2026 publication states that the list contains **1,458 scheduled fixtures**.

This is fundamentally different evidence from Source Version 1 results.

It describes the **scheduled fixture population** rather than reconstructing racing from completed race occurrences.

The annual Fixture List therefore provides the natural baseline source for a future fixture layer.

**Sources:**

British Horseracing Authority, *Full Year*  
https://www.britishhorseracing.com/racing/fixtures/full-year/

British Horseracing Authority, *BHA publishes 2026 fixture list*  
Published: 6 August 2025  
https://www.britishhorseracing.com/press_releases/bha-publishes-2026-fixture-list/


### Evidence 2 — The annual Fixture List is not sufficient on its own

Fixtures can subsequently be:

- transferred;
- rescheduled;
- added;
- abandoned;
- altered;
- or have their race programmes changed.

For example, the three Brighton fixtures transferred in April and May 2026 existed in the scheduled programme before being moved to Great Yarmouth, Bath and Windsor.

The annual schedule therefore represents a **fixture-list state**, not necessarily the final history of every fixture.

A governed fixture layer would need to preserve subsequent administrative changes rather than overwrite the original schedule.

**Source:** British Horseracing Authority, *BHA confirms the transfer of Brighton's next three fixtures*  
Published: 20 April 2026  
https://www.britishhorseracing.com/press_releases/bha-confirms-the-transfer-of-brightons-next-three-fixtures/


### Evidence 3 — Racing Admin carries operational fixture and race-programme detail

BHA fixture-change announcements repeatedly direct participants to **Racing Admin** for the full or updated race programme and race conditions.

For example, after transferring the Brighton fixtures, the BHA stated that full programme details would follow on Racing Admin.

Other BHA announcements similarly direct users to Racing Admin's **Fixtures and Races** material after additional, transferred or amended fixtures are created.

This indicates that Racing Admin forms part of the BHA's operational fixture-management evidence rather than merely being another results source.

**Supporting sources:**

British Horseracing Authority, *BHA confirms the transfer of Brighton's next three fixtures*  
Published: 20 April 2026  
https://www.britishhorseracing.com/press_releases/bha-confirms-the-transfer-of-brightons-next-three-fixtures/

British Horseracing Authority, *BHA confirms additional fixture at Wolverhampton on Thursday 9 April and the rescheduling of two races from Chelmsford's Good Friday fixture*  
Published: 2 April 2026  
https://www.britishhorseracing.com/press_releases/bha-confirms-additional-fixture-at-wolverhampton-on-thursday-9-april-and-the-rescheduling-of-two-races-from-chelmsfords-good-friday-fixture/


### Evidence 4 — Historical fixture evidence may require direct BHA retrieval

The BHA's Full Year page states that details of older fixtures can be requested from the BHA Racing Department.

This matters because the Inside Rails source period begins in 2015.

A complete historical fixture layer should therefore not assume that whatever annual files happen to remain publicly downloadable on the current website constitute the full authoritative archive.

Historical acquisition would need to be treated as a separate governed research task.

**Source:** British Horseracing Authority, *Full Year*  
https://www.britishhorseracing.com/racing/fixtures/full-year/


### Minimum evidence required for a future fixture layer

A defensible future fixture model would need to preserve at least:

- the original scheduled fixture;
- the source and version of the Fixture List in which it appears;
- original scheduled date;
- original scheduled racecourse;
- fixture code/type/session where officially supplied;
- subsequent date changes;
- subsequent venue transfers;
- additional or replacement fixture creation;
- abandonment or cancellation status where established;
- race-programme changes where material;
- provenance for every administrative change;
- linkage to realised race occurrences only after that relationship has been established.

If the BHA exposes a durable official fixture identifier through an authoritative source, that identifier should also be preserved.

No such identifier should be invented from date, racecourse or race times.

## Question 12 — What we found

A genuine BHA fixture layer is **possible in principle**, but it requires a different evidence base from Database v4's completed-race data.

The appropriate evidence architecture would be:

1. **official BHA Fixture List** — baseline scheduled fixture population;
2. **BHA/Racing Admin operational evidence** — programme and fixture details;
3. **dated BHA change evidence** — transfers, rescheduling, additions, abandonments and other amendments;
4. **Inside Rails realised race occurrences** — linked only after the administrative fixture has been independently established.

The important modelling direction is therefore:

> fixture evidence → fixture identity → realised race assignment

not:

> completed races → infer fixture identity

For the historical 2015–2026 Inside Rails period, obtaining and governing the relevant BHA fixture-list history would be a separate database-enhancement project.

Study 04 does not require that project in order to answer its bounded conceptual question.

### Question 12 — Historical availability boundary

The BHA's public website preserves annual Fixture List publication evidence throughout the Inside Rails period back to 2015.

For example:

- 2015 Fixture List publication — 20 October 2014;
- 2016 Fixture List publication — 13 August 2015;
- 2017 Fixture List publication — 1 August 2016;
- 2018 Fixture List publication — 24 July 2017;
- 2019 Fixture List publication — 26 July 2018;
- 2020 Fixture List publication — 6 August 2019.

These publications establish the annual scheduled fixture populations and provide useful fixture-policy information.

However, the current public availability of the underlying machine-readable or downloadable annual files is not uniform.

The 2021 BHA publication explicitly exposes the full Fixture List as both **Excel and PDF**, and later fixture publications similarly provide downloadable files.

The older 2015–2018 publication pages remain publicly accessible, but the currently rendered pages do not expose equivalent Excel/PDF links.

The historical evidence boundary is therefore:

> BHA Fixture List publication evidence is publicly accessible for the full Inside Rails period, but direct public access to the underlying annual fixture files has not yet been established for every year.

A future historical fixture-ingestion project should first inventory the surviving official files before deciding whether any missing years need to be requested directly from the BHA.

**Sources:**

British Horseracing Authority, *British Horseracing's 2015 Fixture List Published*  
Published: 20 October 2014  
https://www.britishhorseracing.com/press_releases/british-horseracings-2015-fixture-list-published/

British Horseracing Authority, *British Horseracing's 2016 Fixture List published*  
Published: 13 August 2015  
https://www.britishhorseracing.com/press_releases/british-horseracings-2016-fixture-list-published/

British Horseracing Authority, *British Racing's full 2021 Fixture List published*  
Published: 26 February 2021  
https://www.britishhorseracing.com/press_releases/british-racings-full-2021-fixture-list-published/

## Question 13 — Does Study 04 require a database change now?

### What the study has established

Study 04 has identified a genuine distinction that matters to the Inside Rails data model:

- a BHA **fixture** is an administrative scheduling object;
- a **meeting** is a context-dependent racing term;
- completed race results do not preserve enough administrative history to reconstruct persistent fixture identity reliably;
- `source date + governed racecourse` supports a useful **source racecourse-date group**, but not a verified fixture;
- a genuine fixture layer could instead be built from BHA fixture-level evidence.

### Required database consequence

No change should be made to accepted Database v4 from the race-results evidence alone.

In particular, Database v4 should **not** gain:

- inferred fixture IDs;
- inferred meeting IDs;
- a `date + racecourse` fixture key;
- inferred session IDs;
- race-to-fixture assignments derived solely from temporal gaps;
- or reconstructed administrative transfer history from realised results.

Those would claim more than the current evidence supports.

### Reusable enhancement identified

Study 04 does identify a legitimate future database enhancement:

> Build a governed Great Britain fixture layer from authoritative BHA fixture-level evidence and then reconcile realised Inside Rails race occurrences to those fixtures.

That enhancement would require a separately governed fixture-source acquisition and reconciliation process.

It should begin by inventorying the available annual BHA Fixture Lists for the Inside Rails period and determining which historical years are available directly and which, if any, require retrieval from the BHA.

Only after the fixture evidence itself is governed should race occurrences be assigned to administrative fixtures.

### Decision

> **Study 04 does not justify changing Database v4 by inference. It identifies a fixture-level database enhancement that must be implemented separately from the reader study using authoritative fixture evidence.**

The existing source race occurrences and governed racecourse identities remain correct.

For analyses performed before a fixture layer exists, `source date + governed racecourse identity` may be used only as the explicitly named **source racecourse-date group**.

## Question 13 — What we found

Study 04 does **not** justify a new fixture or meeting entity in Database v4.

The study has established that:

- the BHA administrative fixture is a real concept;
- Database v4 does not contain enough evidence to reconstruct that administrative identity reliably;
- `source date + governed racecourse identity` remains useful as an explicitly analytical **source racecourse-date group**;
- that grouping must not be presented as a verified BHA fixture or meeting.

### Database decision

Do not add:

- inferred fixture IDs;
- inferred meeting IDs;
- inferred session IDs;
- a `date + racecourse` fixture key;
- race-to-fixture assignments derived from race timing;
- reconstructed transfer or scheduling history inferred from completed results.

A genuine fixture layer is technically possible using BHA fixture-level evidence, but there is no current analytical requirement strong enough to justify building and maintaining it.

### Revisit trigger

Revisit a governed fixture layer only when a specific analysis requires information that completed race results cannot supply, for example:

- scheduled versus realised racing;
- official fixture counts;
- fixture transfers or rescheduling;
- cancellations or abandonments at fixture level;
- planned versus completed race programmes;
- administrative fixture type or allocation;
- analysis explicitly requiring official fixture identity.

Until such a requirement exists, the additional acquisition, reconciliation and governance work would be infrastructure without a sufficiently valuable analytical use.

### Additional consequence discovered at closeout

Study 04 also exposes a separate source-quality opportunity.

Official BHA fixture and results evidence could be used to test whether the Inside Rails source contains every Great Britain race that actually produced an official result.

This is different from constructing administrative fixture identity.

A scheduled fixture absent from the results source would not itself prove a source defect because fixtures and individual races may be abandoned, cancelled, transferred or otherwise changed.

The stronger comparison is:

> official completed BHA race result → corresponding Inside Rails race occurrence

If an officially completed race cannot be reconciled to Source Version 1, that would represent a genuine race-population completeness problem rather than a fixture-modelling issue.

This potentially affects all later Great Britain analysis and is therefore more valuable than building a fixture entity without a current analytical use.

It should be investigated as a **separate bounded database-quality study**, rather than expanding Study 04 further.

### Next bounded investigation identified

> **Are any Great Britain races that officially produced results missing from Source Version 1 / Database v4?**

Study 04 does not attempt that audit. It establishes why BHA fixture/results evidence can provide the independent population reference required to perform it.

# Study 04 — Final conclusion

## Bounded question

**What does British racing mean by a meeting and a fixture, and are they the same thing?**

## Answer

They are **not reliably the same conceptual unit**.

### Fixture

A **fixture** is the more precise BHA administrative scheduling object.

BHA evidence shows that properties of a fixture can change without necessarily creating a different administrative fixture. These include:

- racecourse;
- date;
- race times;
- race programme.

British racing has also permitted two separate fixtures at the same racecourse on the same date.

Therefore:

**`date + racecourse` is not a defensible universal definition of a fixture.**

### Meeting

**Meeting** is a less precise contextual racing term.

BHA usage shows that it may refer to:

- an individual fixture or day's racing; or
- a wider named event containing several dated fixtures.

It should therefore not be assigned one universal technical database identity.

### Raceday and card

**Raceday** is useful ordinary or operational racing terminology, but does not provide a more precise persistent identity than fixture.

**Card** or **race programme** describes the collection or programme of races associated with racing at a fixture. It is not itself fixture identity.

### What Inside Rails can observe

Database v4 primarily preserves realised race occurrences.

It can support the analytical grouping:

**source racecourse-date group**

defined as:

**The set of observed Great Britain race occurrences sharing one source date and one governed racecourse identity.**

That grouping describes the realised source evidence without claiming to reconstruct BHA administrative fixture identity.

### Final modelling rule

Inside Rails should distinguish:

**BHA administrative fixture**

from:

**realised source racecourse-date group**

The former requires fixture-level administrative evidence.

The latter can be derived transparently from Database v4.

The earlier Phase 3 proposal for a persistent **source meeting occurrence** should therefore not be implemented unchanged.

### Database decision

No fixture, meeting or session identity should be added to Database v4 as a result of this study.

A genuine BHA fixture layer should be reconsidered only if a later analytical question actually requires administrative fixture identity.

### Important follow-up discovered by the study

The BHA fixture/results evidence may nevertheless have a more important use.

It can provide an independent external population against which to test whether Source Version 1 contains every Great Britain race that actually produced an official result.

That is a source-completeness question, not a fixture-modelling question, and is escalated to the next bounded investigation.

## Reader-facing report

### Executive conclusion

A British racing **fixture** is not simply "all the races at one racecourse on one date".

It is an administrative scheduled object whose venue, date, race times and programme can change while the BHA may continue to treat it as the same fixture.

The word **meeting** is less precise. It can describe one fixture or day's racing, but it can also describe a larger named event spanning several fixture days.

### Core evidence

The most informative evidence came from boundary cases.

BHA evidence showed that:

- a named multi-day meeting can contain separately dated fixtures;
- two fixtures can occur at the same racecourse on the same date;
- fixtures can be transferred between racecourses;
- fixtures can be rescheduled;
- race times can be changed substantially;
- races within the programme can be altered.

The Inside Rails data then provided an important contrast.

For verified 2026 transfer cases, Database v4 records the races at the racecourse where they were actually staged. It does not preserve the administrative history establishing that those fixtures had originally belonged to another racecourse.

### Interpretation

Completed race results answer:

**What racing actually produced results here?**

They do not necessarily answer:

**What was the identity and administrative history of the scheduled BHA fixture?**

Those are different questions.

### Confidence

Confidence is **high** in the distinction between:

- administrative fixture;
- contextual meeting terminology;
- realised source racecourse-date grouping.

### Limitations

Study 04 does not:

- reconstruct every historical BHA fixture;
- establish a persistent official fixture identifier;
- reconstruct cancelled or wholly abandoned fixtures;
- reconstruct complete scheduled race programmes from completed results;
- create fixture or session boundaries from race-time gaps;
- prove that two completed fixtures never occurred at one racecourse/date during the Inside Rails source period.

Large temporal gaps between completed races were specifically shown to be unreliable fixture boundaries because abandonment or void races can create artificial gaps.

### Practical implication

For most current Inside Rails analysis, races may be grouped transparently by source date and governed racecourse as a:

**source racecourse-date group**

without pretending that this represents verified BHA fixture identity.

A full historical administrative fixture layer should only be built if a later question genuinely needs it.

### Next action

Study 04 exposed a more important validation question:

**Are any Great Britain races that officially produced results missing from Source Version 1 / Database v4?**

That question is moved into a separate bounded database-quality investigation.